# Empirical Analysis of Consumer Price Dynamics (CPI/HICP)

## 1. Theoretical Framework

The Consumer Price Index (CPI), harmonized across the European Union as the HICP, measures the changes over time in the prices of consumer goods and services acquired by households. It constitutes the primary gauge of macroeconomic inflation.

Let $P_{i,t}$ denote the price of item $i$ at time $t$, and $w_i$ its respective weight in the consumption basket. The aggregate price level $P_t$ is computed as a Laspeyres-type index:

$$P_t = \sum_{i=1}^{N} w_i \cdot P_{i,t}$$

The annual inflation rate $\pi_t$ is consequently derived as the year-over-year logarithmic return or discrete percentage change of the index.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%load_ext sql
%sql duckdb:///:memory:

In [ ]:
%%sql
INSTALL httpfs;
LOAD httpfs;

In [ ]:
%%sql cpi_data <<
-- Extracting Annual Average Rate of Change (RCH_A_AVG) for All-Items HICP (CP00)
-- Using the updated Eurostat dataset: prc_hicp_ainr (ECOICOP ver. 2)
WITH eurostat_raw AS (
    SELECT 
        CAST(TIME_PERIOD AS INTEGER) AS year,
        CASE geo
            WHEN 'DE' THEN 'Germany'
            WHEN 'FR' THEN 'France'
            WHEN 'IT' THEN 'Italy'
            WHEN 'ES' THEN 'Spain'
        END AS sovereign_state,
        CAST(OBS_VALUE AS DOUBLE) AS inflation_rate
    FROM read_csv_auto('https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/prc_hicp_ainr/A.RCH_A_AVG.CP00.DE+FR+IT+ES?format=SDMX-CSV')
    WHERE OBS_VALUE IS NOT NULL
)
SELECT * 
FROM eurostat_raw 
ORDER BY year ASC, sovereign_state ASC;

## 2. Historical Trajectories of Inflation Rates
Visualizing the temporal evolution of CPI across primary Eurozone economies to identify aggregate structural shocks.

In [ ]:
# Convert DuckDB ResultSet to Pandas and Pivot for Time Series plotting
df_raw = cpi_data.DataFrame()
df_ts = df_raw.pivot(index='year', columns='sovereign_state', values='inflation_rate')

# Filter for recent decades for clarity (e.g., 2000 onwards)
df_ts = df_ts[df_ts.index >= 2000]

fig, ax = plt.subplots(figsize=(14, 7))

for country in df_ts.columns:
    ax.plot(df_ts.index, df_ts[country], linewidth=2.5, label=country)

# Formatting
ax.axhline(0, color='black', linewidth=1, linestyle='-') # Deflation line
ax.axhline(2, color='red', linewidth=1.5, linestyle='--', alpha=0.6, label='ECB Target (2%)')

ax.set_title('Historical Inflation Trajectories (HICP)\nMajor Eurozone Economies', fontweight='bold', pad=15)
ax.set_xlabel('Fiscal Year', fontweight='bold')
ax.set_ylabel('Annual Inflation Rate (%)', fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(title='Sovereign State', loc='upper left', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

## 3. Cross-Sectional Comparison
Isolating the most recent data point to conduct a static comparative analysis of the monetary deterioration across the selected states.

In [ ]:
# Identify the most recent year in the dataset
latest_year = df_raw['year'].max()
df_latest = df_raw[df_raw['year'] == latest_year].sort_values(by='inflation_rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(df_latest['sovereign_state'], df_latest['inflation_rate'], width=0.6, color='steelblue', edgecolor='black')

# Annotate values on top of bars
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

# Formatting
ax.set_title(f'Cross-Sectional Inflation Comparison ({latest_year})', fontweight='bold', pad=15)
ax.set_ylabel('Annual Inflation Rate (%)', fontweight='bold')
ax.axhline(0, color='black', linewidth=1.2)
ax.grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()